# Hard-hat triage dashboard
Loads the committed weights and the artifacts written by `train.ipynb`. Nothing is trained, downloaded or
re-selected at startup. Run with `uv run python main.py app`.

In [ ]:
"""Dashboard for the frozen, committed YOLO11s hard-hat model."""
from __future__ import annotations

import inspect
import json
import os
from pathlib import Path

import gradio as gr
import numpy as np
import pandas as pd
from matplotlib.figure import Figure
from PIL import Image, ImageDraw, ImageFont
from ultralytics import YOLO

# Works under `main.py app` (nbconvert -> script) and when run cell by cell in VS Code.
ROOT = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd().resolve()
ARTIFACTS = ROOT / "artifacts"

M = json.loads((ARTIFACTS / "metrics.json").read_text(encoding="utf-8"))
VAL = pd.read_csv(ARTIFACTS / "val_cases.csv")
TEST = pd.read_csv(ARTIFACTS / "test_cases.csv")
HEALTH = pd.read_csv(ARTIFACTS / "dataset_health.csv")
SEARCH = pd.read_csv(ARTIFACTS / "review_threshold_search_validation.csv")
MODEL = YOLO(str(ROOT / "weights" / "best.pt"))

COST = dict(M["cost_euros"])
NAMES = M["class_names"]
HEAD, HELMET = M["class_ids"]["head"], M["class_ids"]["helmet"]
SINGLE, LOW0, HIGH0 = M["single_threshold"], M["low_threshold"], M["high_threshold"]
BASE0 = M["base_rate_assumed"]
FLOOR = M["inference_conf"]  # lowest score the model ever reports (0.001)

PASS, REVIEW, FLAG = 0, 1, 2

# Palette: one meaning per colour, used everywhere.
C_PASS, C_REVIEW, C_FLAG = "#2a9d8f", "#e0a100", "#d6452d"
C_MISS, C_FALSE, C_CHECK, C_ACT = "#d6452d", "#8d99ae", "#e0a100", "#2a9d8f"
INK = "#374151"


def policy_actions(df, low, high, triage=True):
    s = df["unsafe_score"].to_numpy(dtype=float)
    if not 0 <= low <= high <= 1:
        raise ValueError("Need 0 <= lower <= upper <= 1")
    if not triage:
        return np.where(s >= low, FLAG, PASS).astype(int)
    actions = np.where(s >= high, FLAG, np.where(s < low, PASS, REVIEW))
    # No head AND no helmet evidence: a PASS would be an unsupported claim.
    return np.where(df["no_evidence"].to_numpy(dtype=bool), REVIEW, actions).astype(int)


def evaluate_actions(df, actions, prevalence, costs, miss=0.0):
    """Expected cost per 1,000 images, re-weighted from the sample to an assumed prevalence.

    `miss` is the share of unsafe images a human reviewer fails to catch (0 = perfect, as in train.ipynb).
    """
    truth = df["unsafe_gt"].to_numpy(dtype=int)
    counts = np.array([[int(((truth == i) & (actions == j)).sum()) for j in (PASS, REVIEW, FLAG)]
                       for i in (0, 1)])
    rates = counts / counts.sum(axis=1, keepdims=True)
    neg, pos = 1 - prevalence, prevalence
    parts = {
        "missed": 1000 * pos * (rates[1, PASS] + rates[1, REVIEW] * miss) * costs["fn"],
        "false_alarm": 1000 * neg * rates[0, FLAG] * costs["fp"],
        "checks": 1000 * (neg * rates[0, REVIEW] + pos * rates[1, REVIEW]) * costs["review"],
        "intervention": 1000 * pos * (rates[1, FLAG] + rates[1, REVIEW] * (1 - miss)) * costs["tp"],
        "other": 1000 * neg * rates[0, PASS] * costs["tn"],
    }
    return {
        "cost": float(sum(parts.values())),
        "parts": {k: float(v) for k, v in parts.items()},
        "review_share": float(neg * rates[0, REVIEW] + pos * rates[1, REVIEW]),
        "missed_rate": float(rates[1, PASS] + rates[1, REVIEW] * miss),
        "counts": counts,
        "rates": rates,
    }


def strategy_costs(prevalence, costs, miss):
    return {
        "Do nothing (always pass)": 1000 * prevalence * costs["fn"],
        "Person checks every image": 1000 * (costs["review"] + prevalence * ((1 - miss) * costs["tp"] + miss * costs["fn"])),
    }


# The artifacts and the dashboard must agree, or the numbers on screen are not the pipeline's numbers.
assert M.get("decision_policy_version", 0) >= 2, "Outdated artifacts: re-run train.ipynb."
_check = evaluate_actions(TEST, policy_actions(TEST, LOW0, HIGH0), BASE0, COST)
assert abs(_check["cost"] - M["test_hybrid"]["cost_per_1000"]) < 1e-6, "metrics.json disagrees with the dashboard maths."

CASES, SECONDS, BUDGET, FTE = M["cases_per_day"], M["review_seconds"], M["review_minutes_available"], M["fte_minutes_per_day"]

CSS = """
html {overflow-y: scroll;}
.gradio-container {max-width: 1200px !important; width: 100% !important; margin: auto !important;}
#controls {position: sticky; top: 12px; align-self: flex-start;}
.kpis {display:grid; grid-template-columns:repeat(4,minmax(0,1fr)); gap:12px; margin:4px 0 10px;}
.kpi {border-left:4px solid #cbd5e1; padding:2px 0 2px 12px;}
.kpi .v {font-size:26px; font-weight:700; line-height:1.2;}
.kpi .l {font-size:13px; font-weight:600;}
.kpi .s {font-size:12px; opacity:.7;}
.kpi.good {border-left-color:#2a9d8f;} .kpi.warn {border-left-color:#e0a100;} .kpi.bad {border-left-color:#d6452d;}
.badge {display:inline-block; padding:6px 16px; border-radius:6px; font-weight:700; font-size:20px; color:#fff;}
.tbl {border-collapse:collapse; width:100%; font-size:13px;}
.tbl th {text-align:left; font-weight:600; border-bottom:2px solid #94a3b8; padding:6px 10px;}
.tbl td {border-bottom:1px solid #cbd5e1; padding:6px 10px;}
.tbl .r {text-align:right;} .tbl tr.hl td {font-weight:700;}
.note {font-size:12.5px; opacity:.75; line-height:1.5;}
@media (max-width: 900px) {.kpis {grid-template-columns:repeat(2,minmax(0,1fr));} #controls {position:static;}}
"""

## Figures and summaries

In [ ]:
def tint(hex_colour: str, share: float) -> str:
    """Opaque tint of a colour over white; dark text stays readable on it."""
    c = np.array([int(hex_colour[i:i + 2], 16) for i in (1, 3, 5)], float)
    t = 0.15 + 0.85 * float(np.clip(share, 0, 1))
    return "#%02x%02x%02x" % tuple((255 - (255 - c) * t).astype(int))


def new_fig(w, h):
    fig = Figure(figsize=(w, h), dpi=100, facecolor="white")
    return fig, fig.add_subplot(111)


def draw_grid(ax, counts, rates, kinds, col_labels, row_labels, fs=15):
    """Coloured cells with a count and its share of the row."""
    for r in range(counts.shape[0]):
        for c in range(counts.shape[1]):
            ax.add_patch(plt_rect(c, r, tint(kinds[r][c], rates[r, c])))
            dark = rates[r, c] > 0.6
            ax.text(c, r - 0.08, f"{counts[r, c]:,}", ha="center", va="center", fontsize=fs, fontweight="bold", color="white" if dark else "#111827")
            ax.text(c, r + 0.24, (f"{rates[r, c]:.1%}" if 0 < rates[r, c] < 0.01 else f"{rates[r, c]:.0%}") + " of row", ha="center", va="center", fontsize=9, color="white" if dark else "#374151")
    ax.set_xlim(-.5, counts.shape[1] - .5)
    ax.set_ylim(counts.shape[0] - .5, -.5)
    ax.set_xticks(range(counts.shape[1]), col_labels, fontsize=11)
    ax.set_yticks(range(counts.shape[0]), row_labels, fontsize=11)
    ax.xaxis.tick_top()
    ax.tick_params(length=0)
    for sp in ax.spines.values():
        sp.set_visible(False)


def plt_rect(c, r, colour):
    from matplotlib.patches import Rectangle
    return Rectangle((c - .48, r - .46), .96, .92, facecolor=colour, edgecolor="white", linewidth=2)


def fig_confusion(counts, rates):
    fig, ax = new_fig(6.4, 3.3)
    n = counts.sum(axis=1)
    draw_grid(ax, counts, rates, [[C_PASS, C_REVIEW, C_FLAG], [C_FLAG, C_REVIEW, C_PASS]],
              ["Pass", "Person checks", "Flag"],
              [f"No exposed head\n{n[0]} images", f"Exposed head\n{n[1]} images"])
    fig.subplots_adjust(left=0.24, right=0.98, top=0.88, bottom=0.04)
    return fig


LOG_GRID = np.unique(np.round(np.geomspace(FLOOR, 1.0, 140), 5))
TICKS = [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1]


def fig_cost_curve(low, high, prevalence, costs, miss):
    lows = LOG_GRID[LOG_GRID <= high]
    val = np.array([evaluate_actions(VAL, policy_actions(VAL, t, high), prevalence, costs, miss)["cost"] for t in lows])
    test = np.array([evaluate_actions(TEST, policy_actions(TEST, t, high), prevalence, costs, miss)["cost"] for t in lows])
    alone = np.array([evaluate_actions(VAL, policy_actions(VAL, t, t, False), prevalence, costs)["cost"] for t in LOG_GRID])
    everyone = strategy_costs(prevalence, costs, miss)["Person checks every image"]
    sub_low = min(max(LOW0, FLOOR), high)
    sub = evaluate_actions(VAL, policy_actions(VAL, sub_low, high), prevalence, costs, miss)["cost"]

    fig, ax = new_fig(7.2, 4.1)
    ax.plot(LOG_GRID, alone, color="#94a3b8", lw=2, ls="--", label="Model alone (validation)")
    ax.plot(lows, test, color=C_ACT, lw=1.6, ls=":", label="Pass / check / flag (test)")
    ax.plot(lows, val, color=C_ACT, lw=3, label="Pass / check / flag (validation)")
    ax.axhline(everyone, color=C_CHECK, lw=1.6, ls="--", label="Person checks every image")
    ax.axvline(max(low, FLOOR), color="#111827", lw=1.4, label="Your pass boundary")
    ax.plot([sub_low], [sub], marker="*", ms=14, color="#111827", ls="", label="Submitted policy")
    ax.set_xscale("log")
    ax.set_xlim(FLOOR, 1)
    ax.set_xticks(TICKS, [str(t) for t in TICKS])
    ax.minorticks_off()
    ax.set_ylim(0, max(2500.0, 3.2 * float(val.min()), 1.6 * everyone))
    ax.yaxis.set_major_formatter(lambda v, _: f"€{v:,.0f}")
    ax.set_xlabel(f"Pass boundary (scores below it are passed; flag boundary fixed at {high:.2f})", fontsize=10)
    ax.set_ylabel("Expected cost per 1,000 images", fontsize=10)
    ax.grid(alpha=.25)
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=2, fontsize=9, frameon=False)
    fig.subplots_adjust(left=0.13, right=0.98, top=0.97, bottom=0.34)
    return fig


def fig_object_confusion():
    cm = np.asarray(BM[FT]["object_confusion"])
    rates = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    kinds = [[C_PASS if r == c else C_FLAG for c in range(4)] for r in range(4)]
    fig, ax = new_fig(6.2, 4.4)
    for r in range(4):
        for c in range(4):
            if r == 3 and c == 3:
                continue
            ax.add_patch(plt_rect(c, r, tint(kinds[r][c], rates[r, c])))
            ax.text(c, r, f"{cm[r, c]:,}", ha="center", va="center", fontsize=13, color="white" if rates[r, c] > 0.6 else "#111827")
    labels = NAMES + ["none"]
    ax.set_xlim(-.5, 3.5); ax.set_ylim(3.5, -.5)
    ax.set_xticks(range(4), labels, fontsize=11); ax.set_yticks(range(4), labels, fontsize=11)
    ax.xaxis.tick_top(); ax.tick_params(length=0)
    ax.set_xlabel("What the model predicted", fontsize=10, labelpad=12); ax.xaxis.set_label_position("top")
    ax.set_ylabel("What is actually there", fontsize=10)
    for sp in ax.spines.values():
        sp.set_visible(False)
    fig.subplots_adjust(left=0.16, right=0.98, top=0.80, bottom=0.03)
    return fig


def html_table(df: pd.DataFrame, right=(), highlight=()):
    head = "".join(f"<th class='{'r' if c in right else ''}'>{c}</th>" for c in df.columns)
    body = "".join(
        f"<tr class='{'hl' if i in highlight else ''}'>" +
        "".join(f"<td class='{'r' if c in right else ''}'>{v}</td>" for c, v in zip(df.columns, row)) + "</tr>"
        for i, row in enumerate(df.itertuples(index=False)))
    return f"<table class='tbl'><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>"


def strategies_html(prevalence, costs, miss, hybrid, alone):
    base = strategy_costs(prevalence, costs, miss)
    rows = [("Do nothing", base["Do nothing (always pass)"], 0.0),
            ("Model decides alone", alone["cost"], 0.0),
            ("Person checks every image", base["Person checks every image"], CASES * SECONDS / 60),
            ("Model + person (current boundaries)", hybrid["cost"], hybrid["review_share"] * CASES * SECONDS / 60)]
    df = pd.DataFrame([[n, f"€{c:,.0f}", f"{m:.0f}", "Yes" if m <= BUDGET else "No"] for n, c, m in rows],
                      columns=["Way of working", "Cost per 1,000 images", "Reviewer min/day", f"Fits {BUDGET} min?"])
    return html_table(df, right=tuple(df.columns[1:]), highlight=(3,))


def kpi_html(res, minutes, prevalence, n_pos):
    spare = BUDGET - minutes
    missed = res["missed_rate"]
    return f"""<div class="kpis">
      <div class="kpi"><div class="l">Expected cost</div><div class="v">€{res['cost']:,.0f}</div>
        <div class="s">per 1,000 images, {prevalence:.0%} unsafe</div></div>
      <div class="kpi warn"><div class="l">Sent to a person</div><div class="v">{res['review_share']:.1%}</div>
        <div class="s">about {res['review_share'] * CASES:.0f} of {CASES} images a day</div></div>
      <div class="kpi {'good' if spare >= 0 else 'bad'}"><div class="l">Reviewer time</div><div class="v">{minutes:.0f} min</div>
        <div class="s">{'%.0f spare of %d' % (spare, BUDGET) if spare >= 0 else '%.0f over the %d available' % (-spare, BUDGET)} min a day</div></div>
      <div class="kpi {'good' if missed < 0.02 else 'warn' if missed < 0.05 else 'bad'}"><div class="l">Unsafe images missed</div><div class="v">{missed:.1%}</div>
        <div class="s">{int(round(res['rates'][1, PASS] * n_pos))} of {n_pos} in the test set passed</div></div>
    </div>"""


def clean_low(value, high):
    """The pass boundary box can be empty while someone is typing; keep it inside [floor, flag boundary]."""
    value = FLOOR if value is None else float(value)
    return float(min(max(value, FLOOR), high))


def refresh(low_in, high, prev_pct, fn_cost, miss_pct):
    high = float(high)
    low = clean_low(low_in, high)
    prevalence, miss = prev_pct / 100, miss_pct / 100
    costs = {**COST, "fn": float(fn_cost)}
    res = evaluate_actions(TEST, policy_actions(TEST, low, high), prevalence, costs, miss)
    alone = evaluate_actions(TEST, policy_actions(TEST, SINGLE, SINGLE, False), prevalence, costs)
    minutes = res["review_share"] * CASES * SECONDS / 60
    c = res["counts"]
    verdict = (f"**Pass** below {low:.4f}. **Person checks** from {low:.4f} to {high:.3f}. **Flag** at {high:.3f} or above. "
               f"Of {int(c.sum())} test photos: {int(c[:, 0].sum())} passed, {int(c[:, 1].sum())} checked, {int(c[:, 2].sum())} flagged. "
               f"Cost and workload assume {prevalence:.0%} unsafe images; the counts are raw.")
    return (verdict, kpi_html(res, minutes, prevalence, int(c[1].sum())), fig_confusion(c, res["rates"]),
            fig_cost_curve(low, high, prevalence, costs, miss), strategies_html(prevalence, costs, miss, res, alone))


def reset_controls():
    return max(LOW0, FLOOR), HIGH0, BASE0 * 100, COST["fn"], 0.0

## Photo inspection and model results

In [ ]:
def _font(size: int):
    try:
        return ImageFont.load_default(size=size)  # Pillow >= 10.1
    except TypeError:
        return ImageFont.load_default()


def inference_once(image):
    """Runs the network once per upload. Moving a slider afterwards only redraws."""
    if image is None:
        return None
    rgb = image.convert("RGB")
    r = MODEL.predict(source=rgb, imgsz=M["imgsz"], conf=M["inference_conf"], iou=M["nms_iou"],
                      max_det=300, verbose=False)[0]
    return {"image": rgb, "ids": r.boxes.cls.cpu().numpy().astype(int).tolist(),
            "scores": r.boxes.conf.cpu().numpy().tolist(), "coords": r.boxes.xyxy.cpu().numpy().tolist()}


def render_inference(cached, low_in, high):
    if cached is None:
        return None, "<p class='note'>Upload a construction-site photo to run the model.</p>"
    high = float(high)
    low = clean_low(low_in, high)
    ids, scores, coords = cached["ids"], cached["scores"], cached["coords"]
    head = max((s for c, s in zip(ids, scores) if c == HEAD), default=0.0)
    helmet = max((s for c, s in zip(ids, scores) if c == HELMET), default=0.0)
    if max(head, helmet) < M["evidence_conf"]:
        label, colour, why = "Person checks", C_REVIEW, "No head or helmet is clear enough to pass the photo."
    elif head >= high:
        label, colour, why = "Flag", C_FLAG, "Strong evidence of an exposed head."
    elif head < low:
        label, colour, why = "Pass", C_PASS, "No head candidate scores above the pass boundary."
    else:
        label, colour, why = "Person checks", C_REVIEW, "Some evidence of an exposed head, not enough to flag it."

    img = cached["image"].copy()
    draw, font = ImageDraw.Draw(img), _font(max(14, img.width // 55))
    width = max(2, img.width // 400)
    palette = {"head": C_FLAG, "helmet": C_PASS, "person": "#4384bd"}
    top_head = max((i for i, c in enumerate(ids) if c == HEAD), key=lambda i: scores[i], default=None)
    for i, (c, s, box) in enumerate(zip(ids, scores, coords)):
        is_driver = i == top_head and s >= low
        if s < M["evidence_conf"] and not is_driver:
            continue
        x1, y1, x2, y2 = box
        col = palette.get(NAMES[c].lower(), "#697b8c")
        draw.rectangle([x1, y1, x2, y2], outline=col, width=width + (1 if is_driver else 0))
        tag = f"{NAMES[c]} {s:.2f}" + (" (decides)" if is_driver else "")
        tw = draw.textlength(tag, font=font)
        draw.rectangle([x1, max(0, y1 - font.size - 4), x1 + tw + 6, max(font.size + 4, y1)], fill=col)
        draw.text((x1 + 3, max(0, y1 - font.size - 3)), tag, fill="white", font=font)
    badge = (f"<span class='badge' style='background:{colour}'>{label}</span> <span style='margin-left:10px'>{why}</span>"
             f"<p class='note'>Top head score {head:.3f}, top helmet score {helmet:.3f}. Scores are model confidence, not probabilities. "
             f"Boxes at 0.25 or above are drawn, plus the top head box if it is above the pass boundary.</p>")
    return img, badge


BM = M["benchmarks"]
FT, COCO, NONE = "YOLO11s fine-tuned", "YOLO11s COCO zero-shot (person mapped only)", "Predict nothing"


def class_table():
    rows = []
    for r in BM[FT]["per_class"]:
        coco = next(x for x in BM[COCO]["per_class"] if x["class"] == r["class"])
        rows.append([r["class"], f"{r['gt_instances']:,}", f"{r['precision_at_0.25']:.2f}", f"{r['recall_at_0.25']:.2f}",
                     f"{r['AP50']:.3f}", f"{r['mAP50_95']:.3f}", f"{coco['AP50']:.3f}", "0.000"])
    rows.append(["mean of three", "", "", "", f"{BM[FT]['AP50']:.3f}", f"{BM[FT]['mAP50_95']:.3f}", f"{BM[COCO]['AP50']:.3f}", "0.000"])
    cols = ["Class", "Test boxes", "Precision", "Recall", "AP50", "mAP50-95", "AP50 COCO zero-shot", "AP50 predict nothing"]
    return html_table(pd.DataFrame(rows, columns=cols), right=tuple(cols[1:]), highlight=(len(rows) - 1,))


def health_table():
    d = HEALTH
    out = pd.DataFrame({
        "Split": d["split"], "Images": d["images"].map("{:,}".format),
        "Head boxes": d["head_instances"].map("{:,}".format), "Helmet boxes": d["helmet_instances"].map("{:,}".format),
        "Person boxes": d["person_instances"].map("{:,}".format),
        "Images with an exposed head": d["head_positive_fraction"].map("{:.1%}".format)})
    return html_table(out, right=tuple(out.columns[1:]))


def cost_table():
    c = COST
    rows = [["Missed unsafe image", f"€{c['fn']:,.2f}", "Passed with an exposed head in view"],
            ["False alarm", f"€{c['fp']:,.2f}", "Flagged, nothing wrong"],
            ["Intervention on a real case", f"€{c['tp']:,.2f}", "Also paid when a person confirms it"],
            ["Correct pass", f"€{c['tn']:,.2f}", "No cost"],
            ["Human check", f"€{c['review']:,.2f}", f"{SECONDS} seconds per image"]]
    return html_table(pd.DataFrame(rows, columns=["Outcome", "Cost", "Meaning"]), right=("Cost",))


NOTE = (f"Volume {CASES} images a day, {SECONDS} s per check, {BUDGET} reviewer minutes available "
        f"({BUDGET * 60 / (CASES * SECONDS):.0%} of images), one full-time reviewer = {FTE} min. "
        f"Assumed unsafe share {BASE0:.0%}; the dataset has {M['dataset_test_base_rate']:.0%} in test. "
        "Reviewers are assumed right every time unless you move the reviewer miss-rate slider.")

## Interface

In [ ]:
# Gradio 5 takes theme/css on Blocks; Gradio 6 takes them on launch().
MODERN_GRADIO = "css" in inspect.signature(gr.Blocks.launch).parameters
THEME = gr.themes.Soft(primary_hue="teal", neutral_hue="slate")
STYLE_ARGS = {} if MODERN_GRADIO else {"theme": THEME, "css": CSS}

with gr.Blocks(title="Hard-hat triage dashboard", **STYLE_ARGS) as demo:
    gr.Markdown("## Hard-hat triage dashboard\nEach site photo is passed, checked by a person, or flagged.")
    with gr.Tabs():
        with gr.Tab("Decision policy"):
            with gr.Row(equal_height=False):
                with gr.Column(scale=1, min_width=280, elem_id="controls"):
                    low = gr.Number(value=max(LOW0, FLOOR), step=0.001, precision=5,
                                    label="Pass boundary", info="Scores below it pass. Type a value from 0.001 to 1.")
                    high = gr.Slider(0.1, 1.0, value=HIGH0, step=0.005, label="Flag boundary",
                                     info="Scores at or above it are flagged.")
                    with gr.Accordion("Assumptions", open=False):
                        prev = gr.Slider(1, 40, value=BASE0 * 100, step=1, label="Unsafe images (%)")
                        fn = gr.Slider(25, 1000, value=COST["fn"], step=25, label="Cost of a missed image (€)")
                        miss = gr.Slider(0, 20, value=0, step=0.5, label="Reviewer miss rate (%)")
                    reset = gr.Button("Back to submitted policy")
                with gr.Column(scale=3):
                    verdict = gr.Markdown()
                    kpis = gr.HTML()
                    with gr.Row():
                        with gr.Column(scale=1):
                            gr.Markdown("**Decisions on the test set**")
                            cm_plot = gr.Plot(show_label=False)
                        with gr.Column(scale=1):
                            gr.Markdown("**Cost against threshold**")
                            curve_plot = gr.Plot(show_label=False)
                    strat = gr.HTML()
                    gr.Markdown("Boundaries were chosen on the validation split and evaluated once on the test split. "
                                "Moving the sliders does not retrain or re-select anything.", elem_classes="note")

        with gr.Tab("Check a photo"):
            with gr.Row():
                uploaded = gr.Image(type="pil", label="Photo", height=420)
                annotated = gr.Image(type="pil", label="Detections", height=420, interactive=False)
            verdict_img = gr.HTML("<p class='note'>Upload a construction-site photo to run the model.</p>")
            gr.Markdown("The model runs once per upload. The boundaries are the ones set on the Decision policy tab.", elem_classes="note")

        with gr.Tab("Model results"):
            gr.Markdown("**head** carries the headline number. **person** is shown because it fails: the fine-tuned model "
                        "finds none of its 64 test boxes at score 0.25, and COCO zero-shot finds more.")
            gr.HTML(class_table())
            gr.Markdown("**Object-level confusion matrix, test split.** The bottom-right cell is empty on purpose: "
                        "detection has no countable \"nothing correctly found as nothing\".")
            gr.Plot(value=fig_object_confusion(), show_label=False)
            gr.Markdown(f"Ultralytics' own test numbers: AP50 {M['native_ultralytics_test']['mAP50']:.3f}, mAP50-95 "
                        f"{M['native_ultralytics_test']['mAP50_95']:.3f}. COCO has no head or helmet class, so it scores zero on both.",
                        elem_classes="note")

        with gr.Tab("Data and assumptions"):
            gr.Markdown("**Dataset** (Roboflow Universe, joseph-nelson / hard-hat-workers, version 10)")
            gr.HTML(health_table())
            gr.Markdown(f"{M['exact_cross_split_duplicate_count']} identical files across splits (SHA-256). "
                        "Near-identical video frames cannot be ruled out.", elem_classes="note")
            gr.Markdown("**What each outcome costs**")
            gr.HTML(cost_table())
            gr.Markdown(NOTE, elem_classes="note")

    controls = [low, high, prev, fn, miss]
    outputs = [verdict, kpis, cm_plot, curve_plot, strat]
    demo.load(refresh, controls, outputs, show_progress="hidden")
    for ctrl in controls:
        ctrl.change(refresh, controls, outputs, show_progress="hidden", trigger_mode="always_last")
    reset.click(reset_controls, None, controls)

    cache = gr.State(None)
    uploaded.change(inference_once, uploaded, cache).then(render_inference, [cache, low, high], [annotated, verdict_img])
    for ctrl in (low, high):
        ctrl.change(render_inference, [cache, low, high], [annotated, verdict_img], show_progress="hidden")

if __name__ == "__main__":
    demo.launch(server_name=os.getenv("APP_HOST", "127.0.0.1"), server_port=int(os.getenv("APP_PORT", "7860")),
                share=False, show_error=True,
                **({"theme": THEME, "css": CSS} if MODERN_GRADIO else {}))